# FOI Sentinel — Demand Simulation & Verified-Query Prioritisation

**Purpose (stress-test tool, _not_ a demo asset).** This notebook drives a realistic, weighted mix of
analyst questions at the `FOI_CASE_ANALYTICS` semantic view, reads back what Cortex Analyst actually
generated, then **fingerprints and clusters the generated SQL by intent** to find where the model is
*unstable* — i.e. the same business question produces materially different SQL. Those unstable, high-demand
intents are the ones that most need an **AI verified query (VQR)** to pin the definition.

It also tangentially exercises the semantic model + search that the app surfaces, so running it warms and
validates the model.

**Pipeline**
1. Canonical **weighted question set** (realistic FOI/EIR/SAR ops demand) with phrasing variants.
2. **Fire the demand** — loop-call the Cortex Analyst REST API in-session (no PAT) via requests + session token.
3. **Read the request log** — `SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_V`.
4. **Fingerprint + cluster** generated SQL by intent; detect definitional drift.
5. **Amplify to target weights** → weighted demand distribution + frequency graph.
6. **Prioritise 'verify next'** — high-demand, high-drift intents not already covered by a verified query.


## 1 · Config

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Execution context (Container Runtime does not set database/schema automatically).
session.sql('USE DATABASE FOI').collect()
session.sql('USE SCHEMA FOI_SENTINEL_V2').collect()

# The semantic view under test, and how it is named in the request log.
SEMANTIC_VIEW  = 'FOI.FOI_SENTINEL_V2.FOI_CASE_ANALYTICS'
MODEL_LOG_NAME = 'FOI.FOI_SENTINEL_V2.FOI_CASE_ANALYTICS'  # SEMANTIC_MODEL_NAME in CORTEX_ANALYST_REQUESTS_V
ANALYST_PATH   = '/api/v2/cortex/analyst/message'
CALL_TIMEOUT_S = 30

# Safety guard: don't fire more than this many calls in one run.
MAX_CALLS = 60
# Variants per intent to actually fire (>=1). Drift shows up across phrasings, so 2 is a good default.
VARIANTS_PER_INTENT = 2

# Host the in-notebook REST call targets; the EAI network rule must allow this exact host.
ACCOUNT_HOST = session.connection.host
print('Config loaded. Target semantic view:', SEMANTIC_VIEW)
print('Account host (must be allowed by the EAI network rule):', ACCOUNT_HOST)


## 2 · Canonical weighted question set

Each **intent** carries a `weight` = its share of real FOI-team questioning demand (weights are relative and
are normalised below). Deadline/overdue and open-backlog questions dominate a real information-governance
team's day, so they carry the most weight. Each intent has phrasing **variants** — drift is exposed by asking
the *same* thing different ways.


In [ ]:
# intent_key: (weight, [phrasing variants])
QUESTION_SET = {
    'overdue_open_count':        (0.16, [
        'How many overdue cases do we have right now?',
        'How many open cases are past their statutory deadline, excluding synthetic test cases?',
        'Count of overdue open requests today',
    ]),
    'overdue_by_regime':         (0.06, [
        'How many overdue cases by regime?',
        'Break down overdue open cases across FOI, EIR and SAR',
    ]),
    'overdue_by_department':     (0.06, [
        'Which owning departments have the most overdue cases?',
        'Overdue open cases grouped by owning department',
    ]),
    'open_count':                (0.10, [
        'How many open cases are there?',
        'Total number of live requests excluding synthetic test cases',
    ]),
    'open_by_regime':            (0.06, [
        'How many open cases by regime?',
        'Open caseload split by FOI, EIR and SAR',
    ]),
    'atrisk_red_count':          (0.10, [
        'How many open cases are at risk?',
        'How many live cases are RAG red?',
    ]),
    'amber_count':               (0.04, [
        'How many open cases are amber?',
        'Count of live cases at amber risk',
    ]),
    'answered_in_time_rate':     (0.08, [
        'What proportion of closed cases were answered in time?',
        'On-time compliance rate for completed requests',
    ]),
    'answered_in_time_by_regime':(0.04, [
        'On-time performance by regime',
        'Share answered within the statutory deadline for FOI vs EIR vs SAR',
    ]),
    'avg_working_days':          (0.04, [
        'What is the average number of working days used on closed cases?',
        'Mean working days to close a request',
    ]),
    'outcome_breakdown':         (0.05, [
        'Break down closed cases by outcome',
        'How many cases were disclosed, partially disclosed or refused?',
    ]),
    'complex_open_cases':        (0.04, [
        'Which open cases are the most complex?',
        'List live cases with the highest complexity',
    ]),
    'high_priority_band':        (0.03, [
        'How many open cases are in the high priority band?',
        'Count of live high-priority requests',
    ]),
    'vexatious_count':           (0.02, [
        'How many cases are flagged vexatious?',
        'Number of requests marked as vexatious',
    ]),
    'caseload_by_officer':       (0.05, [
        'What is the open caseload per assigned officer?',
        'How many live cases does each officer hold?',
    ]),
    'sar_open_count':            (0.03, [
        'How many open SAR cases are there?',
        'Count of live subject access requests',
    ]),
    'published_count':           (0.02, [
        'How many cases have been published to the disclosure log?',
        'Number of requests published',
    ]),
    'negative_sentiment_cases':  (0.02, [
        'Which cases have the most negative requester sentiment?',
        'List cases with the lowest sentiment score',
    ]),
}

import pandas as pd
_w = pd.DataFrame([(k, v[0], len(v[1])) for k, v in QUESTION_SET.items()],
                  columns=['intent', 'weight', 'n_variants'])
_w['weight_norm'] = (_w['weight'] / _w['weight'].sum()).round(4)
_w = _w.sort_values('weight', ascending=False).reset_index(drop=True)
print('Intents:', len(QUESTION_SET), '| total phrasing variants:',
      sum(len(v[1]) for v in QUESTION_SET.values()))
_w


## 3 · Fire the demand (Cortex Analyst REST loop, in-session)

Workspaces notebooks run on **Container Runtime**, where the `_snowflake` helper is unavailable. We call the
Cortex Analyst REST API with `requests` + the notebook's own **session token** (no PAT). Each call is logged
automatically to `CORTEX_ANALYST_REQUESTS_V`; we capture each `request_id` to read back exactly this run's rows.

**Prerequisite for firing:** Container Runtime needs an External Access Integration to reach the account host.
A one-time EAI (`FOI_ANALYST_SELFCALL_EAI`) exists for this account — attach it under **Notebook settings ->
External access**, then re-run. If no EAI is attached, this cell **degrades gracefully to analyze-only**: it
skips firing and the rest of the notebook analyses whatever is already in the request log.


In [ ]:
import requests, json, time

# Flatten the weighted set into the calls we will actually fire (VARIANTS_PER_INTENT each).
calls = []
for intent, (weight, variants) in QUESTION_SET.items():
    for q in variants[:VARIANTS_PER_INTENT]:
        calls.append({'intent': intent, 'weight': weight, 'question': q})
calls = calls[:MAX_CALLS]

def ask_analyst(question):
    url = f'https://{ACCOUNT_HOST}{ANALYST_PATH}'
    headers = {
        'Authorization': f'Snowflake Token="{session.connection.rest.token}"',
        'Content-Type': 'application/json',
        'Accept': 'application/json',
    }
    body = {'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
            'semantic_view': SEMANTIC_VIEW}
    r = requests.post(url, headers=headers, json=body, timeout=CALL_TIMEOUT_S)
    data = r.json()
    # Session-token quirk: an expired token returns HTTP 200 with error code 390112.
    if isinstance(data, dict) and str(data.get('code')) == '390112':
        raise RuntimeError('Session token expired - reconnect the kernel and re-run')
    req_id = data.get('request_id') if isinstance(data, dict) else None
    gen_sql = None
    if isinstance(data, dict):
        for item in data.get('message', {}).get('content', []):
            if item.get('type') == 'sql':
                gen_sql = item.get('statement')
    return r.status_code, req_id, gen_sql

# Preflight probe: if outbound is blocked (no EAI attached), degrade to analyze-only.
FIRING_ENABLED = True
results = []
try:
    _s, _rid, _sql = ask_analyst(calls[0]['question'])
    results.append({**calls[0], 'status': _s, 'request_id': _rid, 'inline_sql': _sql})
    print(f'Probe OK (status {_s}). Firing the remaining {len(calls)-1} calls...')
except Exception as e:
    FIRING_ENABLED = False
    print('DEMAND FIRING DISABLED -', type(e).__name__, str(e)[:160])
    print('Attach FOI_ANALYST_SELFCALL_EAI under Notebook settings -> External access, then re-run.')
    print('Continuing in ANALYZE-ONLY mode against existing CORTEX_ANALYST_REQUESTS_V rows.')

if FIRING_ENABLED:
    for i, c in enumerate(calls[1:], 2):
        try:
            status, req_id, gen_sql = ask_analyst(c['question'])
        except Exception as e:
            status, req_id, gen_sql = ('ERR', None, f'(call error: {e})')
        results.append({**c, 'status': status, 'request_id': req_id, 'inline_sql': gen_sql})
        print(f"  [{i}/{len(calls)}] {c['intent']:<26} status={status} req_id={str(req_id)[:8]}")
        time.sleep(0.3)

fired = pd.DataFrame(results) if results else pd.DataFrame(columns=['intent','weight','question','status','request_id','inline_sql'])
run_request_ids = fired['request_id'].dropna().tolist() if len(fired) else []
print(f'\nCaptured {len(run_request_ids)} request_ids; '
      f"{int((fired['status']==200).sum()) if len(fired) else 0} returned 200.")
fired[['intent','question','status','request_id']] if len(fired) else 'No calls fired (analyze-only mode).'


## 4 · Read the request log

Pull the ground-truth **generated SQL** for exactly the request_ids this run produced. (The view can lag a
few seconds; re-run this cell if some rows are missing.)


In [ ]:
# Dynamic IN-list of the request_ids we just fired (uuids we generated — safe to interpolate).
if run_request_ids:
    id_list = ','.join("'" + rid.replace("'", "") + "'" for rid in run_request_ids)
    where = f'REQUEST_ID IN ({id_list})'
else:
    # Fallback (analyze-only): everything logged against this model in the last 7 days.
    where = (f"SEMANTIC_MODEL_NAME = '{MODEL_LOG_NAME}' "
             "AND TIMESTAMP >= DATEADD('day', -7, CURRENT_TIMESTAMP())")

log = session.sql(f'''
    SELECT REQUEST_ID, LATEST_QUESTION, RESPONSE_STATUS_CODE,
           TO_VARCHAR(TABLES_REFERENCED) AS TABLES_REFERENCED,
           GENERATED_SQL, TIMESTAMP
    FROM SNOWFLAKE.LOCAL.CORTEX_ANALYST_REQUESTS_V
    WHERE {where}
    ORDER BY TIMESTAMP DESC
''').to_pandas()
print(f'Log rows for this run: {len(log)}')
log[['REQUEST_ID','LATEST_QUESTION','RESPONSE_STATUS_CODE']].head(20)


## 5 · Fingerprint & cluster the generated SQL

We reduce each generated statement to a **logic fingerprint** (strip the projection CTE, the trailing
`request_id` comment, aliases and whitespace) and extract **definitional features** — how each query chose to
define *overdue* and *open*. Within one intent, more than one fingerprint = the model is unstable and the
definition is not pinned.


In [ ]:
import re, hashlib

def _norm(s):
    s = s or ''
    s = re.sub(r'--\s*generated by cortex analyst.*', ' ', s, flags=re.I|re.S)
    s = s.replace(';', ' ')
    s = re.sub(r'__v_case', 'v_case', s, flags=re.I)
    s = s.lower()
    return re.sub(r'\s+', ' ', s).strip()

def logic_core(s):
    s = _norm(s)
    s = re.sub(r'with\s+v_case\s+as\s*\(.*?\)\s*', '', s, flags=re.S)  # drop projection CTE
    s = re.sub(r'\bas\s+[a-z_]+_count\b', '', s)                       # normalise count aliases
    return s.strip()

def fingerprint(s):
    core = logic_core(s)
    return hashlib.md5(core.encode()).hexdigest()[:10] if core else '(empty)'

def features(s):
    n = _norm(s)
    f = {}
    if re.search(r'wd_remaining\s*<\s*0', n): f['overdue_def'] = 'WD_REMAINING<0 (verified)'
    elif re.search(r'statutory_deadline\s*<\s*current_date', n): f['overdue_def'] = 'STATUTORY_DEADLINE<CURRENT_DATE (drift)'
    else: f['overdue_def'] = '-'
    if re.search(r"status\s*=\s*'open'", n): f['open_def'] = "STATUS='OPEN' (verified)"
    elif re.search(r'not\s+status\s+in\s*\(', n): f['open_def'] = 'NOT STATUS IN(...) (drift)'
    elif re.search(r'closed_date\s+is\s+null', n): f['open_def'] = 'CLOSED_DATE IS NULL (drift)'
    else: f['open_def'] = '-'
    f['synthetic_excluded'] = bool(re.search(r'is_synthetic\s*=\s*false', n))
    gb = re.search(r'group by\s+(.+?)(\s+order by|$)', n)
    f['group_by'] = gb.group(1).strip()[:60] if gb else '-'
    return f

# Join the log back to the intents we fired (by request_id).
intent_by_id = {r['request_id']: r['intent'] for r in results if r.get('request_id')}
log['intent'] = log['REQUEST_ID'].map(intent_by_id).fillna('(unmatched)')
log['fingerprint'] = log['GENERATED_SQL'].map(fingerprint)
feat = log['GENERATED_SQL'].map(features).apply(pd.Series)
analysis = pd.concat([log[['intent','LATEST_QUESTION','fingerprint','RESPONSE_STATUS_CODE']], feat], axis=1)
analysis.rename(columns={'LATEST_QUESTION':'question'}, inplace=True)
analysis


## 6 · Stability per intent (how many distinct SQL shapes each intent produced)

In [ ]:
stability = (analysis.groupby('intent')
             .agg(n_calls=('fingerprint','size'),
                  distinct_fingerprints=('fingerprint','nunique'),
                  overdue_defs=('overdue_def', lambda s: sorted(set(x for x in s if x!='-'))),
                  open_defs=('open_def', lambda s: sorted(set(x for x in s if x!='-'))))
             .reset_index())
stability['unstable'] = stability['distinct_fingerprints'] > 1
# attach weight
wmap = {k: v[0] for k, v in QUESTION_SET.items()}
stability['weight'] = stability['intent'].map(wmap).fillna(0)
stability = stability.sort_values(['unstable','weight'], ascending=[False, False]).reset_index(drop=True)
stability


## 7 · Amplify to target weights → weighted demand distribution

A handful of real calls stands in for thousands of production questions. We **amplify** each intent's observed
behaviour by its demand weight, so the chart reflects the mix an IG team would actually generate. Bars are
split into demand that lands on a **stable** (single-fingerprint) intent vs an **unstable** one.


In [ ]:
import matplotlib.pyplot as plt

tot_w = sum(v[0] for v in QUESTION_SET.values())
stability['weight_norm'] = stability['weight'] / tot_w
plot_df = stability.sort_values('weight_norm', ascending=True)
colors = ['#c1121f' if u else '#2a9d8f' for u in plot_df['unstable']]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(plot_df['intent'], plot_df['weight_norm'], color=colors)
ax.set_xlabel('Simulated share of FOI questioning demand')
ax.set_title('Weighted demand by intent  (red = unstable SQL / definition not pinned)')
for y, (w, u, d) in enumerate(zip(plot_df['weight_norm'], plot_df['unstable'], plot_df['distinct_fingerprints'])):
    ax.text(w + 0.002, y, f"{w*100:.1f}%" + (f'  ({d} shapes)' if u else ''), va='center', fontsize=8)
plt.tight_layout(); plt.show()

unstable_demand = plot_df.loc[plot_df['unstable'], 'weight_norm'].sum()
print(f'Share of simulated demand hitting an UNSTABLE intent: {unstable_demand*100:.1f}%')


## 8 · 'Verify next' prioritisation

Rank intents by **impact of pinning them** = demand weight × instability, excluding intents already covered by
an `AI_VERIFIED_QUERIES` clause on the semantic view. The top rows are where a new verified query buys the most
consistency for the app + agent.


In [ ]:
# Pull the questions already covered by verified queries on the semantic view.
ddl = session.sql(f"SELECT GET_DDL('SEMANTIC_VIEW', '{SEMANTIC_VIEW}') AS D").to_pandas()['D'][0]
verified_qs = re.findall(r"question\s+'([^']+)'", ddl, flags=re.I)
print('Existing verified queries:', verified_qs)

def already_covered(intent):
    # crude token-overlap match between the intent's canonical question and any verified question
    canon = QUESTION_SET.get(intent, (0, ['']))[1][0].lower()
    ctok = set(re.findall(r'[a-z]+', canon))
    for vq in verified_qs:
        vtok = set(re.findall(r'[a-z]+', vq.lower()))
        if ctok and len(ctok & vtok) / len(ctok) >= 0.5:
            return True
    return False

prio = stability.copy()
prio['covered'] = prio['intent'].map(already_covered)
prio['impact'] = prio['weight_norm'] * (prio['distinct_fingerprints'] - 1).clip(lower=0)
prio = prio[(~prio['covered']) & (prio['distinct_fingerprints'] > 1)]
prio = prio.sort_values('impact', ascending=False).reset_index(drop=True)
print('\nVERIFY NEXT (uncovered + unstable, ranked by demand x instability):')
prio[['intent','weight_norm','distinct_fingerprints','overdue_defs','open_defs','impact']]


## 9 · Acting on the result

For each **verify-next** intent, pick the *correct* definition (the verified one, e.g. overdue =
`STATUS='OPEN' AND WD_REMAINING < 0`), agree it with the data steward, and add it to the semantic view.
`AI_VERIFIED_QUERIES` must appear **before** the legacy `WITH EXTENSION (CA=...)` clause:

```sql
CREATE OR ALTER SEMANTIC VIEW FOI.FOI_SENTINEL_V2.FOI_CASE_ANALYTICS
  ... tables / facts / dimensions ...
  AI_VERIFIED_QUERIES (
    OVERDUE_BY_REGIME AS (
      QUESTION 'How many overdue cases by regime?'
      SQL 'SELECT regime, COUNT(*) AS overdue_open_cases FROM V_CASE
           WHERE status = ''OPEN'' AND is_synthetic = FALSE AND wd_remaining < 0
           GROUP BY regime'
    )
  )
  WITH EXTENSION (CA='...');
```

Then re-run this notebook: pinned intents should collapse to a **single fingerprint** and drop off the
verify-next list. That closed loop — simulate demand → find drift → pin with a VQR → re-simulate — is how the
semantic model is hardened before it is shown in the app.
